In [1]:
from data_fetcher import DataFetcher, DataFetcherConfig
from data_processor import OptionGammaProcessor, OptionGammaConfig
from utils import export_parquet_sample_to_csv, log_parquet_inventory
from pathlib import Path

In [ ]:
cfg = DataFetcherConfig(
        wrds_username="sniperw",
        data_dir="data",
        start_date="2024-01-01",
        end_date="2024-12-31",
        start_year=2024,
        end_year=2024,
        file_type="parquet",
        compression="zstd",
        replace=True,
        crsp_permno_chunk_size=500,
        optionm_secid_chunk_size=5,
        min_abs_prc=None,
        include_ticker_fallback=False,
        keep_intermediate_csv=False,
    )

"""cfg = DataFetcherConfig(
        wrds_username="sniperw",
        data_dir="data",
        start_date="2000-01-01",
        end_date="2025-12-31",
        start_year=2000,
        end_year=2025,
        file_type="parquet",
        compression="zstd",
        replace=False,
        crsp_permno_chunk_size=500,
        optionm_secid_chunk_size=25,
        min_abs_prc=5.0,
        include_ticker_fallback=False,
        keep_intermediate_csv=False,
    )"""
fetcher = DataFetcher(cfg)

In [7]:
fetcher.run_identifier_pipeline()

In [2]:
log_parquet_inventory(Path("data"))
log_parquet_inventory(Path("results/option_gamma"))

WindowsPath('results/option_gamma/parquet_inventory_log.txt')

In [2]:
fetcher.fetch_crsp_dsf(combine_final=True)

CRSP DSF months:   0%|          | 0/12 [00:00<?, ?month/s]

2024-01:   0%|          | 0/9 [00:00<?, ?chunk/s]

2024-02:   0%|          | 0/9 [00:00<?, ?chunk/s]

2024-03:   0%|          | 0/9 [00:00<?, ?chunk/s]

2024-04:   0%|          | 0/9 [00:00<?, ?chunk/s]

...

2024-09:   0%|          | 0/9 [00:00<?, ?chunk/s]

2024-10:   0%|          | 0/9 [00:00<?, ?chunk/s]

2024-11:   0%|          | 0/9 [00:00<?, ?chunk/s]

2024-12:   0%|          | 0/9 [00:00<?, ?chunk/s]

WindowsPath('data/crsp_daily_common_2024_2024.parquet')

In [2]:
fetcher.build_crsp_liquidity_panel()

WindowsPath('data/crsp_liquidity_panel_2024_2024.parquet')

In [3]:
fetcher.build_top_liquid_stock_universe(top_pct=0.001)

WindowsPath('data/crsp_top_liquid_universe_2024_2024_0p001.parquet')

In [3]:
fetcher.build_linked_secid_file_from_top_liquid()

WindowsPath('data/linked_secids_top_liquid_2024_2024.parquet')

In [3]:
txt_path = fetcher.export_secids_to_txt(
    parquet_file=("data/linked_secids_top_liquid_2024_2024.parquet")
)
print(txt_path)

data\linked_secids_top_liquid_2024_2024.txt


In [5]:
fetcher.fetch_opprcd()

opprcd month-chunks:   0%|          | 0/24 [00:00<?, ?chunk/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WindowsPath('data/opprcd_linked_2024_2024.parquet')

In [2]:
fetcher.fetch_secprd()

secprd month-chunks:   0%|          | 0/24 [00:00<?, ?chunk/s]

WindowsPath('data/secprd_linked_2024_2024.parquet')

In [7]:
export_parquet_sample_to_csv("data/opprcd_linked_2024_2024.parquet", n=2000, order_by="secid")

WindowsPath('data/opprcd_linked_2024_2024_head_2000.csv')

In [8]:
fetcher.close()

In [1]:
cfg = OptionGammaConfig(
    option_parquet="data/opprcd_linked_2024_2024.parquet",
    spot_parquet="data/secprd_linked_2024_2024.parquet",
    output_dir="results/option_gamma",
    start_date="2024-01-01",
    end_date="2024-12-31",
    secids=None,
    spot_price_field="close",
    require_spot=True,
    memory_limit="16GB",
    threads=8,
)

processor = OptionGammaProcessor(cfg)


[2026-03-19 04:25:38] [INFO] OptionGammaProcessor_2016078251312 - Registering option parquet: data\opprcd_linked_2024_2024.parquet
[2026-03-19 04:25:38] [INFO] OptionGammaProcessor_2016078251312 - Registering spot parquet: data\secprd_linked_2024_2024.parquet


In [2]:
diag = processor.run_diagnostics()
print(diag["spot_join_coverage_report"])

[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.00s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.08s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.01s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.08s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.33s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.00s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.00s
[2026-03-19 04:25:45] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.14s


   n_option_rows  n_matched_spot  n_missing_spot  matched_ratio
0        7195783       7195783.0             0.0            1.0


In [3]:
processor.run_baseline_pipeline(
    export_contract_gex="results/option_gamma/contract_gex.parquet",
    export_underlying_daily="results/option_gamma/underlying_gex_daily.parquet",
)

[2026-03-19 04:26:12] [INFO] OptionGammaProcessor_2016078251312 - Exporting query to parquet: results\option_gamma\contract_gex.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-03-19 04:26:15] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 2.82s
[2026-03-19 04:26:15] [INFO] OptionGammaProcessor_2016078251312 - Exporting query to parquet: results\option_gamma\underlying_gex_daily.parquet
[2026-03-19 04:26:16] [INFO] OptionGammaProcessor_2016078251312 - Query finished in 0.68s


{'contract_gex': WindowsPath('results/option_gamma/contract_gex.parquet'),
 'underlying_daily': WindowsPath('results/option_gamma/underlying_gex_daily.parquet')}

In [4]:
processor.close()

[2026-03-19 04:26:36] [INFO] OptionGammaProcessor_2016078251312 - Closing DuckDB connection


In [2]:
export_parquet_sample_to_csv("results/option_gamma/contract_gex.parquet", n=1000, order_by="date")
export_parquet_sample_to_csv("results/option_gamma/underlying_gex_daily.parquet", frac=1,order_by="date")

WindowsPath('results/option_gamma/underlying_gex_daily_sample_100pct.csv')

In [9]:
fetcher.close()